In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('Toddler Autism dataset July 2018.csv')


In [3]:
# strip whitespace from column names
df.columns = df.columns.str.strip()

### Dropping `Ethnicity`

`Ethnicity` is excluded from the feature set for two reasons:

1. **Fairness** — using race/ethnicity as a raw predictor in a child health-risk model requires a rigorous, documented justification and per-group error analysis. Without that, it's not responsible to leave it in.
2. **It was never a nominal-safe encoding anyway** — a straightforward `LabelEncoder` on a category like `Ethnicity` assigns arbitrary integers (e.g. `White European` → 5, `Hispanic` → 3) that imply a false ordinal relationship the model can exploit spuriously. Fixing that properly (one-hot encoding) would add ~10 sparse columns to an already-small (1,054 row) dataset — not a good trade for a feature with no established clinical link to ASD likelihood.

`Sex`, `Jaundice`, and `Family_mem_with_ASD` are kept — each is binary, so label encoding introduces no false ordinality, and each has real (if modest) support in the clinical literature as a risk covariate.

In [4]:
# drop columns we don't need (see rationale above for Ethnicity)
df.drop(columns=['Case_No', 'Qchat-10-Score', 'Who completed the test', 'Ethnicity'], inplace=True)

In [5]:
# encode target column
df['target'] = df['Class/ASD Traits'].map({'Yes': 1, 'No': 0})
df.drop(columns=['Class/ASD Traits'], inplace=True)

In [6]:
# encode binary categorical columns
cat_cols = ['Sex', 'Jaundice', 'Family_mem_with_ASD']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [7]:
print("Shape after cleaning:", df.shape)
print("\nColumns remaining:", df.columns.tolist())
print("\nTarget distribution:")
print(df['target'].value_counts())
print("\nFirst 5 rows:")
print(df.head())

Shape after cleaning: (1054, 15)

Columns remaining: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'Age_Mons', 'Sex', 'Jaundice', 'Family_mem_with_ASD', 'target']

Target distribution:
target
1    728
0    326
Name: count, dtype: int64

First 5 rows:
   A1  A2  A3  A4  A5  A6  A7  A8  A9  A10  Age_Mons  Sex  Jaundice  \
0   0   0   0   0   0   0   1   1   0    1        28    0         1   
1   1   1   0   0   0   1   1   0   0    0        36    1         1   
2   1   0   0   0   0   0   1   1   0    1        36    1         1   
3   1   1   1   1   1   1   1   1   1    1        24    1         0   
4   1   1   0   1   1   1   1   1   1    1        20    0         0   

   Family_mem_with_ASD  target  
0                    0       0  
1                    0       1  
2                    0       1  
3                    0       1  
4                    1       1  


In [8]:
df.to_csv('cleaned_toddler.csv', index=False)
print("Saved! Shape:", df.shape)

Saved! Shape: (1054, 15)


In [9]:
df_check = pd.read_csv('cleaned_toddler.csv')
print("Loaded back successfully:", df_check.shape)
print(df_check.head(2))


Loaded back successfully: (1054, 15)
   A1  A2  A3  A4  A5  A6  A7  A8  A9  A10  Age_Mons  Sex  Jaundice  \
0   0   0   0   0   0   0   1   1   0    1        28    0         1   
1   1   1   0   0   0   1   1   0   0    0        36    1         1   

   Family_mem_with_ASD  target  
0                    0       0  
1                    0       1  
